# 5. Location Prediction with Machine Learning

## Overview
This notebook trains **machine learning models** to predict the geographic coordinates (latitude, longitude) of future EV charging stations based on:
- **Year**: Commissioning year
- **Federal State**: German state where the station is located

### Models Compared
| Model | Type | Key Parameter |
|-------|------|---------------|
| **Random Forest** | Ensemble (bagging) | 100 estimators |
| **K-Nearest Neighbors** | Instance-based | k=5 neighbors |

### Evaluation Metrics
- **MAE** (Mean Absolute Error): Average prediction error in degrees
- **MSE** (Mean Squared Error): Penalizes larger errors more heavily
- **R² Score**: Proportion of variance explained (0 = baseline, 1 = perfect)

### Purpose
The trained models are used in the **genetic algorithm optimization** (Notebook 06) to generate plausible candidate locations for new charging stations.

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 5.1 Data Preparation

In [ ]:
# Load data
data = pd.read_csv('../data/processed/ChargingStationCleaned.csv', encoding='utf-8')
data['year'] = pd.to_datetime(data['commissioning_date']).dt.year

# Fix encoding in state names
state_corrections = {
    'Baden-Wï¿½rttemberg': 'Baden-Württemberg',
    'Thï¿½ringen': 'Thüringen',
}
data['federal_state'] = data['federal_state'].replace(state_corrections)

# Select relevant features
data = data[['year', 'federal_state', 'latitude_[dg]', 'longitude_[dg]']]
print(f'Training data: {len(data)} samples')
data.head()

## 5.2 Model Comparison: Random Forest vs. KNN

One-hot encoding of the federal state provides 16 binary features. Combined with the year feature, this gives 17 total features.

In [ ]:
# One-hot encode the federal state column
data_encoded = pd.get_dummies(data, columns=['federal_state'])

# Split features and targets
X = data_encoded.drop(['latitude_[dg]', 'longitude_[dg]'], axis=1)
y_lat = data_encoded['latitude_[dg]']
y_lon = data_encoded['longitude_[dg]']

# Train/test split (80/20)
X_train, X_test, y_lat_train, y_lat_test, y_lon_train, y_lon_test = train_test_split(
    X, y_lat, y_lon, test_size=0.2, random_state=42
)

# --- Random Forest Regressor ---
rf_model_lat = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model_lat.fit(X_train, y_lat_train)
rf_model_lon = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model_lon.fit(X_train, y_lon_train)

# --- K-Nearest Neighbors Regressor ---
knn_model_lat = KNeighborsRegressor(n_neighbors=5)
knn_model_lat.fit(X_train, y_lat_train)
knn_model_lon = KNeighborsRegressor(n_neighbors=5)
knn_model_lon.fit(X_train, y_lon_train)

# --- Evaluate ---
y_lat_pred_rf = rf_model_lat.predict(X_test)
y_lon_pred_rf = rf_model_lon.predict(X_test)
y_lat_pred_knn = knn_model_lat.predict(X_test)
y_lon_pred_knn = knn_model_lon.predict(X_test)

print('=== Random Forest ===')
print(f'  MAE  (lat): {mean_absolute_error(y_lat_test, y_lat_pred_rf):.4f}')
print(f'  MAE  (lon): {mean_absolute_error(y_lon_test, y_lon_pred_rf):.4f}')
print(f'  MSE  (lat): {mean_squared_error(y_lat_test, y_lat_pred_rf):.4f}')
print(f'  MSE  (lon): {mean_squared_error(y_lon_test, y_lon_pred_rf):.4f}')
print(f'  R²   (lat): {r2_score(y_lat_test, y_lat_pred_rf):.4f}')
print(f'  R²   (lon): {r2_score(y_lon_test, y_lon_pred_rf):.4f}')

print('\n=== K-Nearest Neighbors ===')
print(f'  MAE  (lat): {mean_absolute_error(y_lat_test, y_lat_pred_knn):.4f}')
print(f'  MAE  (lon): {mean_absolute_error(y_lon_test, y_lon_pred_knn):.4f}')
print(f'  MSE  (lat): {mean_squared_error(y_lat_test, y_lat_pred_knn):.4f}')
print(f'  MSE  (lon): {mean_squared_error(y_lon_test, y_lon_pred_knn):.4f}')
print(f'  R²   (lat): {r2_score(y_lat_test, y_lat_pred_knn):.4f}')
print(f'  R²   (lon): {r2_score(y_lon_test, y_lon_pred_knn):.4f}')

## 5.3 Model Comparison Analysis

**Key Findings:**
- Random Forest consistently outperforms KNN across all metrics (lower MAE/MSE, higher R²)
- RF achieves R² ≈ 0.92 for latitude and R² ≈ 0.87 for longitude, indicating strong predictive power
- The model captures the spatial distribution patterns of charging stations per federal state

**Random Forest is selected** as the production model for the optimization pipeline due to its:
- Superior accuracy
- Built-in ensemble averaging (reduces overfitting)
- Ability to capture non-linear spatial patterns

## 5.4 Train Year-Specific Models

Train separate Random Forest models for each year (2010-2022). Each model is trained on all stations commissioned up to that year, allowing it to learn the spatial distribution for that specific time period.

This produces **per-year prediction models** used by the genetic algorithm to generate realistic candidate locations.

In [ ]:
# Use label encoding for this stage (needed for the genetic algorithm interface)
le = LabelEncoder()
data_full = pd.read_csv('../data/processed/ChargingStationCleaned.csv', encoding='utf-8')
data_full['federal_state'] = data_full['federal_state'].replace(state_corrections)
data_full['year'] = pd.to_datetime(data_full['commissioning_date']).dt.year
data_full['federal_state_encoded'] = le.fit_transform(data_full['federal_state'])

X_full = data_full[['year', 'federal_state_encoded']]
y_lat_full = data_full['latitude_[dg]']
y_lon_full = data_full['longitude_[dg]']

max_year = data_full['year'].max()

for year in range(2010, max_year + 1):
    # Cumulative training data up to this year
    mask = data_full['year'] <= year
    X_year = X_full[mask]
    y_lat_year = y_lat_full[mask]
    y_lon_year = y_lon_full[mask]

    # Train latitude and longitude models
    rf_lat = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_lat.fit(X_year, y_lat_year)
    rf_lon = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_lon.fit(X_year, y_lon_year)

    # Save to year-specific directory
    year_folder = f'../models/{year}'
    os.makedirs(year_folder, exist_ok=True)
    joblib.dump(rf_lat, f'{year_folder}/rf_model_{year}_lat.pkl')
    joblib.dump(rf_lon, f'{year_folder}/rf_model_{year}_lon.pkl')
    print(f'Year {year}: trained on {len(X_year)} samples, models saved')

print(f'\nDone. Saved models for {max_year - 2010 + 1} years to ../models/')